In [1]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

# log_dir = PROJECT_ROOT / "src" / "logs"
# log_dir.mkdir(parents=True, exist_ok=True)

# 4. adding src folder to python path
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

# 5. checking imports
try:
    (PROJECT_ROOT / "logs").mkdir(exist_ok=True) 
    
    from database import get_connection
    from pipeline import run_full_pipeline, run_incremental_pipeline
    from logging_utils import setup_logger
    
    print(f"✅ Project Root: {PROJECT_ROOT}")
    print(f"✅ Logs Directory: {log_dir}")
    print("✅ Imports successful")
except Exception as e:
    print(f"❌ Import xətası: {e}")

✅ Project Root: /Users/samil/Desktop/ITSkillsSprintProjects/construction-weather-risk-planner
❌ Import xətası: name 'log_dir' is not defined


In [2]:

(PROJECT_ROOT / "data" / "database").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "data" / "raw").mkdir(parents=True, exist_ok=True)

db_path = PROJECT_ROOT / "data" / "database" / "weather_daily.duckdb"

conn = get_connection(str(db_path))

print(f"✅ Database Path: {db_path}")
print(f"✅ Logs Directory: {PROJECT_ROOT / 'src' / 'logs'}")

✅ Database Path: /Users/samil/Desktop/ITSkillsSprintProjects/construction-weather-risk-planner/data/database/weather_daily.duckdb
✅ Logs Directory: /Users/samil/Desktop/ITSkillsSprintProjects/construction-weather-risk-planner/src/logs


- Created necessary project directories for logs, database, and raw data.
- Defined DuckDB database path for weather data storage.
- Established database connection using `get_connection()`.


In [3]:
try:
    tables = conn.execute("SHOW TABLES").fetchall()
    print("Existing tables:", tables)
except Exception as e:
    print("Tables have not been created yet (they will be created after the full pipeline runs).")

Existing tables: []


The code attempts to list all existing tables in the DuckDB database.

- If tables exist → it prints the available table names.
- If no tables are found → it handles the exception and indicates that tables will be created later in the full pipeline stage.


In [4]:
logger = setup_logger()

logger.info("===== FULL PIPELINE START =====")

df_full = run_full_pipeline(conn, logger)
logger.info("===== FULL PIPELINE END =====")

print("✅ Full pipeline completed successfully!")
display(df_full)

Fetching → Baku
Baku: 1827 rows
Fetching → Ganja
Ganja: 1827 rows
Fetching → Shusha
Shusha: 1827 rows
Fetching → Nakhchivan
Nakhchivan: 1827 rows
Loading Historical CSV: Ganja_historical.csv
Loading Historical CSV: Nakhchivan_historical.csv
Skipping empty file: .gitkeep
Loading Historical CSV: Baku_historical.csv
Loading Historical CSV: Shusha_historical.csv
Loading Forecast CSV: Baku_forecast.csv
Loading Forecast CSV: Ganja_forecast.csv
Loading Forecast CSV: Nakhchivan_forecast.csv
Skipping empty file: .gitkeep
Loading Forecast CSV: Shusha_forecast.csv

══════════════════════════════════════════════════
  Processing raw.weather_daily_historical …
  Rows loaded : 7,308
  Handling missing values …
  Nulls before: 0  →  after: 0
  Flagging outliers (IQR, threshold=1.5) …
  Total outlier flags: 4,174  across 12 columns
  Validating date continuity …

─────────────────────────────────────────────
  City            : Ganja
  Date range      : 2021-05-01 → 2026-05-01
  Expected days   : 1827

{'mode': 'full',
 'historical_rows': 7308,
 'forecast_rows': 64,
 'staging_hist_rows': 7308,
 'staging_forecast_rows': 64,
 'analytics_hist_rows': 7308,
 'analytics_forecast_rows': 64,
 'duration': '0:00:13.435810'}

The logging system is initialized to track the entire workflow.

- The pipeline start is recorded using a logger message.
- The full data pipeline (`run_full_pipeline`) is executed using the database connection.
- Completion of the pipeline is also logged for traceability.


In [5]:
hist_count = conn.execute("SELECT COUNT(*) FROM raw.weather_daily_historical").fetchone()[0]
feat_count = conn.execute("SELECT COUNT(*) FROM analytics.weather_features_historical").fetchone()[0]

print(f" RAW HIST ROWS: {hist_count}")
print(f" ANALYTICS FEATURES ROWS: {feat_count}")

 RAW HIST ROWS: 7308
 ANALYTICS FEATURES ROWS: 7308



- **Raw historical weather data table** (`raw.weather_daily_historical`)
- **Engineered features table** (`analytics.weather_features_historical`)

The output helps verify that:
- The raw dataset has been successfully ingested.
- Feature engineering has produced the expected number of records.


In [6]:
logger.info("===== INCREMENTAL PIPELINE START =====")
df_inc = run_incremental_pipeline(conn, logger)
logger.info("===== INCREMENTAL PIPELINE END =====")

print("✅ Incremental run completed.")
display(df_inc)

Fetching → Baku
⚠️ end_date adjusted from 2026-05-03 to 2026-05-02
⚠️ Rate limit hit. Waiting 60s...
Baku: 1 rows
Fetching → Ganja
⚠️ end_date adjusted from 2026-05-03 to 2026-05-02
Ganja: 1 rows
Fetching → Shusha
⚠️ end_date adjusted from 2026-05-03 to 2026-05-02
Shusha: 1 rows
Fetching → Nakhchivan
⚠️ end_date adjusted from 2026-05-03 to 2026-05-02
Nakhchivan: 1 rows

══════════════════════════════════════════════════
  Processing raw.weather_daily_historical …
  Rows loaded : 7,308
  Handling missing values …
  Nulls before: 0  →  after: 0
  Flagging outliers (IQR, threshold=1.5) …
  Total outlier flags: 4,174  across 12 columns
  Validating date continuity …

─────────────────────────────────────────────
  City            : Ganja
  Date range      : 2021-05-01 → 2026-05-01
  Expected days   : 1827
  Actual days     : 1827
  Missing days    : 0
─────────────────────────────────────────────

─────────────────────────────────────────────
  City            : Nakhchivan
  Date range    

{'mode': 'incremental',
 'rows_added_raw': 4,
 'affected_cities': ['Baku', 'Ganja', 'Shusha', 'Nakhchivan'],
 'skipped_cities': [],
 'raw_hist_total': 7308,
 'raw_forecast_total': 64,
 'staging_hist': 7308,
 'analytics_hist': 7308,
 'analytics_forecast': 64,
 'timestamp': '2026-05-03 16:00:59.629268'}


- The start and end of the pipeline are logged for tracking purposes.
- `run_incremental_pipeline` is called using the existing database connection and logger.
- After execution, a success message confirms that the incremental update has completed.
- The resulting dataframe (`df_inc`) is displayed for validation.


In [7]:
print("Running incremental again...")

df_inc_2 = run_incremental_pipeline(conn, logger)

df_inc_2

Running incremental again...
Fetching → Baku
⚠️ end_date adjusted from 2026-05-03 to 2026-05-02
Baku: 1 rows
Fetching → Ganja
⚠️ end_date adjusted from 2026-05-03 to 2026-05-02
Ganja: 1 rows
Fetching → Shusha
⚠️ end_date adjusted from 2026-05-03 to 2026-05-02
Shusha: 1 rows
Fetching → Nakhchivan
⚠️ end_date adjusted from 2026-05-03 to 2026-05-02
Nakhchivan: 1 rows

══════════════════════════════════════════════════
  Processing raw.weather_daily_historical …
  Rows loaded : 7,308
  Handling missing values …
  Nulls before: 0  →  after: 0
  Flagging outliers (IQR, threshold=1.5) …
  Total outlier flags: 4,174  across 12 columns
  Validating date continuity …

─────────────────────────────────────────────
  City            : Ganja
  Date range      : 2021-05-01 → 2026-05-01
  Expected days   : 1827
  Actual days     : 1827
  Missing days    : 0
─────────────────────────────────────────────

─────────────────────────────────────────────
  City            : Nakhchivan
  Date range      : 2

{'mode': 'incremental',
 'rows_added_raw': 4,
 'affected_cities': ['Baku', 'Ganja', 'Shusha', 'Nakhchivan'],
 'skipped_cities': [],
 'raw_hist_total': 7308,
 'raw_forecast_total': 64,
 'staging_hist': 7308,
 'analytics_hist': 7308,
 'analytics_forecast': 64,
 'timestamp': '2026-05-03 16:01:22.154474'}

In [8]:
print("RAW HIST:",
      conn.execute("SELECT COUNT(*) FROM raw.weather_daily_historical").fetchone())

print("RAW FORECAST:",
      conn.execute("SELECT COUNT(*) FROM raw.weather_daily_forecast").fetchone())

RAW HIST: (7308,)
RAW FORECAST: (64,)


This step checks the number of records in key raw data tables:

- **Historical weather data** (`raw.weather_daily_historical`)
- **Forecast data** (`raw.weather_daily_forecast`)


In [9]:
log_path = PROJECT_ROOT / "logs" / "pipeline.log"

if log_path.exists():
    with open(log_path, "r", encoding="utf-8") as f:
        logs = f.readlines()
    print(f"✅ Log file found at: {log_path}")
    print("\nSon 15 log qeydi:")
    print("".join(logs[-15:]))
else:
    print(f"❌ XƏTA: Log faylı bu yolda tapılmadı: {log_path}")

✅ Log file found at: /Users/samil/Desktop/ITSkillsSprintProjects/construction-weather-risk-planner/logs/pipeline.log

Son 15 log qeydi:
2026-05-03 16:01:22,139 | weather_pipeline | INFO | Running data quality checks...
2026-05-03 16:01:22,153 | weather_pipeline | INFO | row_count → PASS → raw.weather_daily_historical: 7308 rows
2026-05-03 16:01:22,153 | weather_pipeline | INFO | freshness → PASS → raw.weather_daily_historical: 1 days old
2026-05-03 16:01:22,153 | weather_pipeline | INFO | row_count → PASS → raw.weather_daily_forecast: 64 rows
2026-05-03 16:01:22,153 | weather_pipeline | INFO | freshness → PASS → raw.weather_daily_forecast: -15 days old
2026-05-03 16:01:22,153 | weather_pipeline | INFO | null_ratio → PASS → staging.weather_daily_historical: OK
2026-05-03 16:01:22,153 | weather_pipeline | INFO | date_continuity → PASS → staging.weather_daily_historical: OK
2026-05-03 16:01:22,153 | weather_pipeline | INFO | temperature_range → PASS → staging.weather_daily_historical: OK


This step verifies whether the pipeline log file exists and is accessible.

- If the log file is found, its path is displayed.
- The last 15 log entries are printed to review recent pipeline activity.
- If the file is missing, an error message is shown indicating the expected log path.


In [10]:
print("""
================ PIPELINE ARCHITECTURE ================

        WEATHER API
             │
             ▼
     INGESTION LAYER
             │
             ▼
       RAW (DuckDB)
             │
             ▼
     STAGING (CLEANING)
             │
             ▼
   FEATURE ENGINEERING
             │
             ▼
     ANALYTICS LAYER
             │
             ▼
     QUALITY CHECKS
             │
             ▼
        LOGGING SYSTEM

FULL MODE:
- Full rebuild of dataset

INCREMENTAL MODE:
- Only new data ingestion
- Skips if up-to-date

======================================================
""")


================ PIPELINE ARCHITECTURE ================

        WEATHER API
             │
             ▼
     INGESTION LAYER
             │
             ▼
       RAW (DuckDB)
             │
             ▼
     STAGING (CLEANING)
             │
             ▼
   FEATURE ENGINEERING
             │
             ▼
     ANALYTICS LAYER
             │
             ▼
     QUALITY CHECKS
             │
             ▼
        LOGGING SYSTEM

FULL MODE:
- Full rebuild of dataset

INCREMENTAL MODE:
- Only new data ingestion
- Skips if up-to-date




### Pipeline Architecture Overview



- **Weather API** → Data source  
- **Ingestion Layer** → Collects raw data  
- **Raw (DuckDB)** → Stores unprocessed data  
- **Staging Layer** → Data cleaning and preprocessing  
- **Feature Engineering** → Creates analytical features  
- **Analytics Layer** → Final structured dataset for analysis  
- **Quality Checks** → Ensures data correctness and consistency  
- **Logging System** → Tracks all pipeline steps  

### Pipeline Modes

- **Full Mode:** Rebuilds the entire dataset from scratch  
- **Incremental Mode:** Processes only new data and skips existing records if up-to-date  

This architecture ensures scalability, modularity, and efficient data processing.

In [11]:
print("""
RESULT SUMMARY:

✅ Full pipeline executed successfully
✅ Data ingested → cleaned → transformed
✅ Features created
✅ Quality checks executed
✅ Logs written to pipeline.log
✅ Incremental mode handled correctly

TASK 5 COMPLETED SUCCESSFULLY 🚀
""")


RESULT SUMMARY:

✅ Full pipeline executed successfully
✅ Data ingested → cleaned → transformed
✅ Features created
✅ Quality checks executed
✅ Logs written to pipeline.log
✅ Incremental mode handled correctly

TASK 5 COMPLETED SUCCESSFULLY 🚀



In [12]:
conn.close()